In [ ]:
import pandas as pd

In [ ]:
df_2015 = pd.read_csv('2015_복지_사회참여_문화와여가_소득과소비_노동.csv')
df_2017 = pd.read_csv('2017_복지_사회참여_문화와여가_소득과소비_노동.csv')
df_2019 = pd.read_csv('2019_복지_사회참여_문화와여가_소득과소비_노동.csv')
df_2021 = pd.read_csv('2021_복지_사회참여_문화와여가_소득과소비_노동.csv')
df_2023 = pd.read_csv('2023_복지_사회참여_문화와여가_소득과소비_노동.csv')
df_2017

In [ ]:
# 데이터프레임들을 딕셔너리로 묶음
years = {
    "2015": df_2015,
    "2017": df_2017,
    "2019": df_2019,
    "2021": df_2021,
    "2023": df_2023
}

# 각 연도별 피쳐 추출
features = {year: df.columns.tolist() for year, df in years.items()}

# 출력
for year, cols in features.items():
    print(f"{year} features:", cols)


In [ ]:
cols_2015 = set(features["2015"])
cols_2017 = set(features["2017"])
cols_2019 = set(features["2019"])
cols_2021 = set(features["2021"])
cols_2023 = set(features["2023"])

In [ ]:
all_items = sorted(cols_2015 | cols_2017 | cols_2019 | cols_2021 | cols_2023)
compare_df = pd.DataFrame({
    "항목명": all_items,
    "2015": [item in cols_2015 for item in all_items],
    "2017": [item in cols_2017 for item in all_items],
    "2019": [item in cols_2019 for item in all_items],
    "2021": [item in cols_2021 for item in all_items],
    "2023": [item in cols_2023 for item in all_items],
})

# 각 행에서 True 개수 세기
compare_df["True_count"] = (compare_df[["2015", "2017", "2019", "2021", "2023"]]).sum(axis=1)

# 출력 옵션: 열 다 보이도록
# 출력 옵션 늘리기
pd.set_option("display.max_rows", None)      # 행 전부 출력
pd.set_option("display.max_columns", None)  # 열 전부 출력
pd.set_option("display.width", None)        # 줄바꿈 없이 한 줄로 출력
pd.set_option("display.max_colwidth", None) # 셀 안 텍스트 길이 제한 해제

print(compare_df)

                                        항목명   2015   2017   2019   2021  \
0                                     가구가중값   True   True   True   True   
1                                    가구소득코드   True   True   True   True   
2                                    가구원가중값   True   True   True   True   
3                                     가구원번호   True   True   True   True   
4                                    가구일련번호   True   True   True   True   
5                                   가구주관계코드   True   True   True   True   
6                              개인적인간관계만족도코드  False   True   True   True   
7                                    거처종류코드   True   True   True   True   
8                                    계층의식코드   True   True   True   True   
9                        고용안정성_불안사유_코로나관련여부  False  False  False   True   
10                                  고용안정성코드   True   True   True   True   
11                               과거해외여행경험여부   True  False  False  False   
12                       

In [ ]:
compare_df.to_csv("항목명_일치여부비교.csv", index=False, encoding="utf-8-sig")

In [ ]:
import re

def normalize_column(col):
    # 한글+숫자+영문만 남기고, 나머지는 제거
    col = re.sub(r"[^가-힣A-Za-z0-9()_]", "", col)
    # 소문자로 변환
    col = col.lower()
    # 맨 뒷 글자 '코드' 제거
    col = re.sub(r"코드$", "", col)
    return col

delete_blink = [normalize_column(c) for c in compare_df["항목명"]]


In [ ]:
compare_df["항목명"]

In [ ]:
from rapidfuzz import process, fuzz

cols = compare_df["항목명"]
matches = {}
for col in cols:
    match = process.extractOne(col, cols, scorer=fuzz.ratio)
    matches[col] = match

In [ ]:
# 매칭 결과 저장
results = []
for col in compare_df["항목명"]:
    match, score, _ = process.extractOne(col, delete_blink, scorer=fuzz.ratio)
    results.append([col, match, f"{score:.4f}"])

# 데이터프레임으로 변환
match_df = pd.DataFrame(results, columns=["원본컬럼(df1)", "매칭컬럼(df2)", "유사도"])

# CSV로 저장
match_df.to_csv("항목명_매칭결과.csv", index=False, encoding="utf-8-sig")

print("항목명_매칭결과.csv 파일로 저장 완료!")

column_matching.csv 파일로 저장 완료!


In [ ]:
matched_column = {
    "레저시설_스키장이용횟수": "레저시설_스키장스케이트장이용횟수",
    "레저시설이용횟수": "레저시설_이용여부",
    "문화예술스포츠관람_음악연주회횟수": "문화예술스포츠관람_음악연주회콘서트횟수", 
    "분류코드_고용안정성": "분류코드_고용안정성여부",
    "자원봉사활동_재해지역주민돕기시설복구관련횟수": "자원봉사활동_재해지역주민돕기및시설복구관련횟수",
    "자원봉사활동_재해지역주민돕기시설복구관련시간수": "자원봉사활동_재해지역주민돕기및시설복구관련주기",
    "정기기부_현금기부_주기": "정기기부_현금기부주기",
    "정기기부_물품기부_주기": "정기기부_물품기부주기",
    "자원봉사활동_환경보호_연간참여총횟수": "자원봉사활동_환경보전범죄예방관련횟수",
    "자원봉사활동_환경보호_1회평균활동시간수": "자원봉사활동_환경보전범죄예방관련시간수",
    "자원봉사활동_국가지역행사관련시간수": "자원봉사활동_국가및지역행사관련시간수",
    "자원봉사활동_국제협력해외봉사_1회평균활동시간수": "자원봉사활동_국가및지역행사관련시간수",
    "자원봉사활동_국가지역행사관련횟수": "자원봉사활동_국가및지역행사관련횟수",
    "자원봉사활동_국제협력해외봉사_연간참여총횟수": "자원봉사활동_국가및지역행사관련횟수",
    "분류코드_고용안전성": "분류코드_고용안정성여부",
    "기부내용_물품후원단체후원_물품횟수": "기부내용_물품횟수",
    "노후사회적역할기타내용": "노후사회적역할",
    "여가활용불만족사유": "여가활용_불만족사유"
}

In [ ]:
# 특정 열에서 치환
match_df["매칭컬럼(df2)"] = match_df["매칭컬럼(df2)"].replace(matched_column)
match_df = match_df.drop(columns=["유사도"])
match_df

In [ ]:
# 매핑 딕셔너리 생성
mapping_dict = dict(zip(match_df["원본컬럼(df1)"], match_df["매칭컬럼(df2)"]))

# 각 데이터프레임 컬럼명 교체 (덮어쓰기 방식)
df_2015_renamed = df_2015.rename(columns=mapping_dict)
df_2017_renamed = df_2017.rename(columns=mapping_dict)
df_2019_renamed = df_2019.rename(columns=mapping_dict)
df_2021_renamed = df_2021.rename(columns=mapping_dict)
df_2023_renamed = df_2023.rename(columns=mapping_dict)

In [ ]:
# 연도별 데이터프레임을 딕셔너리에 저장
dfs = {
    "2015": df_2015_renamed,
    "2017": df_2017_renamed,
    "2019": df_2019_renamed,
    "2021": df_2021_renamed,
    "2023": df_2023_renamed
}

# CSV로 저장 (UTF-8)
for year, df in dfs.items():
    file_name = f"df_{year}.csv"
    df.to_csv(file_name, index=False, encoding="utf-8-sig")
    print(f"{file_name} 저장 완료 ✅")


df_2015.csv 저장 완료 ✅
